# Phase 1 Final Merge Execution
Execute the Phase 1 merge to combine all components into the final addon

In [1]:
import os
import sys
import json
import shutil
import uuid
import zipfile
from pathlib import Path

# Execute Phase 1 merge directly
BASE_DIR = "/workspaces/Mcai"
PHASE1_DIR = os.path.join(BASE_DIR, "work_phase1")
WORK_MERGE_DIR = os.path.join(BASE_DIR, "work_merge_phase1")
OUTPUT_ADDON = os.path.join(BASE_DIR, "PostApocalypticSurvival_ULTIMATE_v5.0.0-Phase1.mcaddon")

def log(msg):
    print(f"[✓] {msg}", flush=True)

def warn(msg):
    print(f"[!] {msg}", flush=True)

print("="*60)
print("PHASE 1: FINAL MERGE EXECUTION")
print("="*60 + "\n")

print(f"BASE_DIR: {BASE_DIR}")
print(f"PHASE1_DIR exists: {os.path.exists(PHASE1_DIR)}")
print(f"Output will be: {OUTPUT_ADDON}\n")

PHASE 1: FINAL MERGE EXECUTION

BASE_DIR: /workspaces/Mcai
PHASE1_DIR exists: True
Output will be: /workspaces/Mcai/PostApocalypticSurvival_ULTIMATE_v5.0.0-Phase1.mcaddon



In [2]:
# Step 1: Setup directories
log("Creating work directory...")
os.makedirs(WORK_MERGE_DIR, exist_ok=True)
base_addon = os.path.join(BASE_DIR, "PostApocalypticSurvival_UPGRADED.mcaddon")
extract_base = os.path.join(WORK_MERGE_DIR, "base_addon")
output_structure = os.path.join(WORK_MERGE_DIR, "PostApocalypticSurvival_ULTIMATE")

print(f"Base addon exists: {os.path.exists(base_addon)}")
print(f"Base addon size: {os.path.getsize(base_addon) / (1024*1024):.2f} MB")

[✓] Creating work directory...
Base addon exists: True
Base addon size: 39.32 MB


In [3]:
# Step 2: Extract base addon
log("Extracting base addon v4.0.0...")
if os.path.exists(extract_base):
    shutil.rmtree(extract_base)
os.makedirs(extract_base)

with zipfile.ZipFile(base_addon, 'r') as zf:
    zf.extractall(extract_base)

log(f"Extracted to: {extract_base}")
print(f"Contents: {os.listdir(extract_base)}")

[✓] Extracting base addon v4.0.0...
[✓] Extracted to: /workspaces/Mcai/work_merge_phase1/base_addon
Contents: ['PostApocalypticSurvival_UPGRADED']


In [4]:
# Step 3: Setup output structure
log("Setting up output addon structure...")
if os.path.exists(output_structure):
    shutil.rmtree(output_structure)
os.makedirs(output_structure)

# Find BP/RP in extracted addon
bp_source = None
rp_source = None

print("Searching for behavior_pack and resource_pack...")
for item in os.listdir(extract_base):
    item_path = os.path.join(extract_base, item)
    print(f"  {item} (dir: {os.path.isdir(item_path)})")
    if os.path.isdir(item_path):
        if 'behavior' in item.lower():
            bp_source = item_path
        elif 'resource' in item.lower():
            rp_source = item_path

if not bp_source or not rp_source:
    warn("Could not find behavior_pack or resource_pack at root, searching subdirectories...")
    for root, dirs, files in os.walk(extract_base):
        for d in dirs:
            if d == 'behavior_pack':
                bp_source = os.path.join(root, d)
            elif d == 'resource_pack':
                rp_source = os.path.join(root, d)

log(f"Found behavior_pack: {bp_source is not None}")
log(f"Found resource_pack: {rp_source is not None}")

[✓] Setting up output addon structure...
Searching for behavior_pack and resource_pack...
  PostApocalypticSurvival_UPGRADED (dir: True)
[!] Could not find behavior_pack or resource_pack at root, searching subdirectories...
[✓] Found behavior_pack: True
[✓] Found resource_pack: True


In [5]:
# Step 4: Copy base packs to output
bp_dest = os.path.join(output_structure, "behavior_pack")
rp_dest = os.path.join(output_structure, "resource_pack")

log("Copying base packs...")
if bp_source:
    shutil.copytree(bp_source, bp_dest)
    log(f"  Behavior pack copied ({len(os.listdir(bp_dest))} items)")
if rp_source:
    shutil.copytree(rp_source, rp_dest)
    log(f"  Resource pack copied ({len(os.listdir(rp_dest))} items)")

[✓] Copying base packs...
[✓]   Behavior pack copied (11 items)
[✓]   Resource pack copied (12 items)


In [6]:
# Step 5: Add Phase 1 structures (NBT files)
log("Adding Lost Cities structures (10 NBT files)...")
structures_src = os.path.join(PHASE1_DIR, "structures")
structures_dest = os.path.join(bp_dest, "structures")
os.makedirs(structures_dest, exist_ok=True)

nbt_files = []
if os.path.exists(structures_src):
    for nbt_file in os.listdir(structures_src):
        if nbt_file.endswith('.nbt'):
            src = os.path.join(structures_src, nbt_file)
            dst = os.path.join(structures_dest, nbt_file)
            shutil.copy2(src, dst)
            nbt_files.append(nbt_file)
    log(f"  Copied {len(nbt_files)} NBT structures")
    print(f"  Structure files: {sorted(nbt_files)}")

[✓] Adding Lost Cities structures (10 NBT files)...
[✓]   Copied 10 NBT structures
  Structure files: ['bunker_main.nbt', 'city_ruins_01.nbt', 'city_ruins_02.nbt', 'city_ruins_03.nbt', 'metro_station.nbt', 'metro_tunnel_straight.nbt', 'metro_tunnel_turn.nbt', 'military_base_variant1.nbt', 'military_base_variant2.nbt', 'military_base_variant3.nbt']


In [7]:
# Step 6: Add magazine items
log("Adding TACZ magazine items (6 items)...")
items_dest = os.path.join(bp_dest, "items")
os.makedirs(items_dest, exist_ok=True)

ammo_src = os.path.join(PHASE1_DIR, "ammo_system")
mag_files = []
if os.path.exists(ammo_src):
    for mag_file in os.listdir(ammo_src):
        if mag_file.startswith('magazine_') and mag_file.endswith('.json'):
            src = os.path.join(ammo_src, mag_file)
            dst = os.path.join(items_dest, mag_file)
            shutil.copy2(src, dst)
            mag_files.append(mag_file)
    log(f"  Copied {len(mag_files)} magazine items")
    print(f"  Magazine items: {sorted(mag_files)}")

[✓] Adding TACZ magazine items (6 items)...
[✓]   Copied 6 magazine items
  Magazine items: ['magazine_12gauge.json', 'magazine_338lapua.json', 'magazine_556nato.json', 'magazine_762x39.json', 'magazine_9mm.json', 'magazine_grenade.json']


In [8]:
# Step 7: Add ammo system script
log("Adding ammo_system.js...")
scripts_dest = os.path.join(bp_dest, "scripts")
os.makedirs(scripts_dest, exist_ok=True)

ammo_js = os.path.join(ammo_src, "ammo_system.js")
if os.path.exists(ammo_js):
    shutil.copy2(ammo_js, os.path.join(scripts_dest, "ammo_system.js"))
    log("  Copied ammo_system.js")
    js_size = os.path.getsize(ammo_js) / 1024
    print(f"  ammo_system.js size: {js_size:.1f} KB")

[✓] Adding ammo_system.js...
[✓]   Copied ammo_system.js
  ammo_system.js size: 9.8 KB


In [9]:
# Step 8: Add zombie entities + loot tables
log("Adding TZP zombie mutants (4 entities + 4 loot tables)...")
entities_dest = os.path.join(bp_dest, "entities")
loot_dest = os.path.join(bp_dest, "loot_tables")
os.makedirs(entities_dest, exist_ok=True)
os.makedirs(loot_dest, exist_ok=True)

entities_src = os.path.join(PHASE1_DIR, "entities")
entity_count = 0
loot_count = 0
if os.path.exists(entities_src):
    for f in os.listdir(entities_src):
        src = os.path.join(entities_src, f)
        if f.startswith('zombie_') and f.endswith('.json') and '_loot' not in f:
            dst = os.path.join(entities_dest, f)
            shutil.copy2(src, dst)
            entity_count += 1
        elif '_loot.json' in f:
            dst = os.path.join(loot_dest, f)
            shutil.copy2(src, dst)
            loot_count += 1
    log(f"  Copied {entity_count} entity definitions + {loot_count} loot tables")

[✓] Adding TZP zombie mutants (4 entities + 4 loot tables)...
[✓]   Copied 4 entity definitions + 4 loot tables


In [10]:
# Step 9: Update manifests
log("Updating manifests to v5.0.0 ULTIMATE...")

# Generate UUIDs
bp_uuid = str(uuid.uuid4())
rp_uuid = str(uuid.uuid4())

# Update behavior pack manifest
bp_manifest_path = os.path.join(bp_dest, "manifest.json")
if os.path.exists(bp_manifest_path):
    with open(bp_manifest_path, 'r', encoding='utf-8') as f:
        bp_manifest = json.load(f)
    
    bp_manifest["version"] = [5, 0, 0]
    bp_manifest["uuid"] = bp_uuid
    bp_manifest["name"] = "Post-Apocalyptic Survival BP - ULTIMATE"
    
    if "header" in bp_manifest:
        bp_manifest["header"]["version"] = [5, 0, 0]
        bp_manifest["header"]["uuid"] = bp_uuid
    
    with open(bp_manifest_path, 'w', encoding='utf-8') as f:
        json.dump(bp_manifest, f, indent=2, ensure_ascii=False)
    
    log("  Updated behavior_pack manifest to v5.0.0")
    print(f"  BP UUID: {bp_uuid}")

[✓] Updating manifests to v5.0.0 ULTIMATE...
[✓]   Updated behavior_pack manifest to v5.0.0
  BP UUID: e374c310-5f56-4313-a9bb-8e9c3fb5d5a2


In [11]:
# Update resource pack manifest
rp_manifest_path = os.path.join(rp_dest, "manifest.json")
if os.path.exists(rp_manifest_path):
    with open(rp_manifest_path, 'r', encoding='utf-8') as f:
        rp_manifest = json.load(f)
    
    rp_manifest["version"] = [5, 0, 0]
    rp_manifest["uuid"] = rp_uuid
    rp_manifest["name"] = "Post-Apocalyptic Survival RP - ULTIMATE"
    
    if "header" in rp_manifest:
        rp_manifest["header"]["version"] = [5, 0, 0]
        rp_manifest["header"]["uuid"] = rp_uuid
    
    with open(rp_manifest_path, 'w', encoding='utf-8') as f:
        json.dump(rp_manifest, f, indent=2, ensure_ascii=False)
    
    log("  Updated resource_pack manifest to v5.0.0")
    print(f"  RP UUID: {rp_uuid}")

[✓]   Updated resource_pack manifest to v5.0.0
  RP UUID: 7d53df95-454c-4318-95bc-9fb8aeb0287e


In [12]:
# Step 10: Package addon
log("Packaging final .mcaddon file...")

if os.path.exists(OUTPUT_ADDON):
    os.remove(OUTPUT_ADDON)

file_count = 0
with zipfile.ZipFile(OUTPUT_ADDON, 'w', zipfile.ZIP_DEFLATED) as zf:
    for root, dirs, files in os.walk(output_structure):
        for file in files:
            file_path = os.path.join(root, file)
            arcname = os.path.relpath(file_path, output_structure)
            zf.write(file_path, arcname)
            file_count += 1

file_size_mb = os.path.getsize(OUTPUT_ADDON) / (1024 * 1024)
log(f"Created {OUTPUT_ADDON}")
print(f"  File size: {file_size_mb:.2f} MB")
print(f"  Files in archive: {file_count}")

[✓] Packaging final .mcaddon file...
[✓] Created /workspaces/Mcai/PostApocalypticSurvival_ULTIMATE_v5.0.0-Phase1.mcaddon
  File size: 39.28 MB
  Files in archive: 846


In [13]:
# Verify it's a valid ZIP
log("Verifying output ZIP file...")
try:
    with zipfile.ZipFile(OUTPUT_ADDON, 'r') as zf:
        file_list = zf.namelist()
        file_count = len(file_list)
        test_result = zf.testzip()
        
        log(f"  ZIP contains {file_count} files")
        if test_result is None:
            log("  ZIP integrity: ✓ PASSED")
        else:
            print(f"  ⚠ Corrupted file: {test_result}")
        
        # Show some key files
        key_files = [f for f in file_list if 'manifest' in f or 'nbt' in f or 'ammo' in f or 'zombie' in f]
        print(f"\n  Key files in archive:")
        for f in sorted(key_files)[:10]:
            print(f"    - {f}")
except Exception as e:
    print(f"❌ Verification failed: {e}")

[✓] Verifying output ZIP file...
[✓]   ZIP contains 846 files
[✓]   ZIP integrity: ✓ PASSED

  Key files in archive:
    - behavior_pack/entities/apocalypse_zombie.json
    - behavior_pack/entities/crawler_zombie.json
    - behavior_pack/entities/horde_zombie.json
    - behavior_pack/entities/juggernaut_zombie.json
    - behavior_pack/entities/military_zombie.json
    - behavior_pack/entities/nighthunter_zombie.json
    - behavior_pack/entities/tnt_zombie.json
    - behavior_pack/entities/zombie_brute.json
    - behavior_pack/entities/zombie_jockey.json
    - behavior_pack/entities/zombie_soldier.json


In [14]:
# Summary
print("\n" + "="*60)
print("✅ PHASE 1 MERGE COMPLETE")
print("="*60)
print(f"Output File: {OUTPUT_ADDON}")
print(f"File Size: {file_size_mb:.2f} MB")
print(f"Total Files: {file_count}")
print(f"Status: Ready for testing")
print("="*60)

# Verify file exists
print(f"\nFile exists on disk: {os.path.exists(OUTPUT_ADDON)}")
print(f"Is readable: {os.access(OUTPUT_ADDON, os.R_OK)}")
print(f"Actual file size: {os.path.getsize(OUTPUT_ADDON) / (1024*1024):.2f} MB")


✅ PHASE 1 MERGE COMPLETE
Output File: /workspaces/Mcai/PostApocalypticSurvival_ULTIMATE_v5.0.0-Phase1.mcaddon
File Size: 39.28 MB
Total Files: 846
Status: Ready for testing

File exists on disk: True
Is readable: True
Actual file size: 39.28 MB
